## **설정**

In [1]:
## Google Drive Amount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
## Import libaries
import pandas as pd
import numpy as np

import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

In [3]:
## Path Setting
base_path = '/content/drive/MyDrive/CS2/'
input_path = os.path.join(base_path, 'myData')
output_path = os.path.join(base_path, 'hidden_states')

In [4]:
## random seed 고정
import random

def set_seed(val):
    torch.manual_seed(val)
    torch.cuda.manual_seed(val)
    # torch.cuda.manual_seed_all(val)  # 멀티 GPU를 사용하는 경우
    np.random.seed(val)
    random.seed(val)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [5]:
## HS 추출 설정
WINDOW_SIZE = 6                     # 6개월치 패턴을 한 번에 봄
output_dims = [5, 10, 20, 50]       # 추출할 히든 스테이트 차원

## **데이터 불러오기**

In [6]:
## Load the data
macro2018 = pd.read_csv(os.path.join(input_path, 'macro2018.csv')) # 1996-07~2018-12
macro2019 = pd.read_csv(os.path.join(input_path, 'macro2019.csv')) # 1996-07~2019-12
macro2020 = pd.read_csv(os.path.join(input_path, 'macro2020.csv')) # 1996-07~2020-12
macro2021 = pd.read_csv(os.path.join(input_path, 'macro2021.csv')) # 1996-07~2021-12

## **CNN Encoder**

In [7]:
class MultiHeadCNNEncoder(nn.Module):
    def __init__(self, input_dim, output_dims=[5, 10, 20, 50]):
        super(MultiHeadCNNEncoder, self).__init__()
        self.output_dims = output_dims
        self.heads = nn.ModuleList()
        for out_dim in output_dims:
            head = nn.Sequential(
                nn.Conv1d(input_dim, out_dim * 2, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_dim * 2),
                nn.LeakyReLU(0.1),
                nn.Conv1d(out_dim * 2, out_dim, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_dim),
                nn.LeakyReLU(0.1),
                nn.AdaptiveMaxPool1d(1),
                nn.Flatten()
            )
            self.heads.append(head)

    def forward(self, x):
        x = x.permute(0, 2, 1) # (Batch, Seq, Feat) -> (Batch, Feat, Seq)
        results = {}
        for i, head in enumerate(self.heads):
            results[f'hidden_{self.output_dims[i]}'] = head(x)
        return results

In [8]:
# 윈도우 생성 함수
def create_sliding_window(data, dates, window):
    X, D = [], []
    for i in range(len(data) - window + 1):
        X.append(data[i:i+window])
        D.append(dates[i+window-1])  # 마지막 날짜를 window의 대표 날짜로 설정
    return np.array(X), np.array(D)

## **데이터 분할 & HS 추출**

In [9]:
# 처리할 데이터 리스트 (데이터프레임, Test연도)
data_list = [
    (macro2018, 2018),
    (macro2019, 2019),
    (macro2020, 2020),
    (macro2021, 2021)
]

In [10]:
for df_origin, test_year in data_list:
    # 1. 날짜 처리 및 정렬
    df = df_origin.copy()
    df['sasdate'] = pd.to_datetime(df['sasdate'])
    df = df.sort_values('sasdate').reset_index(drop=True)

    # Feature 컬럼 정의 (sasdate 제외)
    feature_cols = [c for c in df.columns if c not in ['sasdate', 'year']] # year가 있다면 제외

    # 2. 구간 설정 (Logic: Test=Target, Valid=Target-3~Target-1, Train=1997~Target-4)
    # 예: 2018 -> Test(2018), Valid(2015,16,17), Train(1997~2014)

    train_end_year = test_year - 4
    valid_start_year = test_year - 3
    valid_end_year = test_year - 1

    print(f"\n======== Processing Data for Test Year {test_year} ========")
    print(f"Train: 1997 ~ {train_end_year}")
    print(f"Valid: {valid_start_year} ~ {valid_end_year}")
    print(f"Test : {test_year}")

    # 3. 데이터 분할 (Split)
    # Train: 1997-01-01부터 시작 (Window 고려하여 앞부분 데이터 필요시 조정 가능, 여기선 1997.1.1 기준)
    train_start_date = pd.Timestamp(1997, 1, 1) - pd.DateOffset(months=WINDOW_SIZE-1)
    train_df = df[(df['sasdate'] >= train_start_date) &
                  (df['sasdate'] <= pd.Timestamp(train_end_year, 12, 31))].copy()

    # Valid: 시작일에서 Window 크기만큼 앞의 데이터를 포함해야 첫 달부터 예측 가능
    valid_start_adj = pd.Timestamp(valid_start_year, 1, 1) - pd.DateOffset(months=WINDOW_SIZE-1)
    valid_df = df[(df['sasdate'] >= valid_start_adj) &
                  (df['sasdate'] <= pd.Timestamp(valid_end_year, 12, 31))].copy()

    # Test: 마찬가지로 Window 크기만큼 앞 데이터 포함
    test_start_adj = pd.Timestamp(test_year, 1, 1) - pd.DateOffset(months=WINDOW_SIZE-1)
    test_df = df[(df['sasdate'] >= test_start_adj) &
                 (df['sasdate'] <= pd.Timestamp(test_year, 12, 31))].copy()

    # 4. 스케일링 (Scaling) - Train 기준으로 fit
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols])

    train_scaled = scaler.transform(train_df[feature_cols])
    valid_scaled = scaler.transform(valid_df[feature_cols])
    test_scaled  = scaler.transform(test_df[feature_cols])

    # 5. 슬라이딩 윈도우 (Windowing)
    X_train, dates_train = create_sliding_window(train_scaled, train_df['sasdate'].values, WINDOW_SIZE)
    X_valid, dates_valid = create_sliding_window(valid_scaled, valid_df['sasdate'].values, WINDOW_SIZE)
    X_test, dates_test   = create_sliding_window(test_scaled, test_df['sasdate'].values, WINDOW_SIZE)

    # 텐서 변환
    tensors = {
        'train': torch.FloatTensor(X_train),
        'valid': torch.FloatTensor(X_valid),
        'test':  torch.FloatTensor(X_test)
    }
    dates_dict = {'train': dates_train, 'valid': dates_valid, 'test': dates_test}

    # 6. 모델 초기화 및 특징 추출 (Random Weights)
    model = MultiHeadCNNEncoder(len(feature_cols), output_dims=output_dims)
    model.eval()

    # 저장 폴더 생성 (hidden_states/Test_2018)
    round_dir = os.path.join(output_path, 'CNN_macro', f'Test_{test_year}')
    os.makedirs(round_dir, exist_ok=True)

    with torch.no_grad():
        for split in ['train', 'valid', 'test']:
            if len(tensors[split]) == 0:
                print(f"Warning: {split} set is empty for year {test_year}")
                continue

            features = model(tensors[split])

            for dim in output_dims:
                feats = features[f'hidden_{dim}'].numpy()
                cols = [f'hs_{dim}_{k}' for k in range(dim)]

                temp_df = pd.DataFrame(feats, columns=cols)
                temp_df.insert(0, 'sasdate', dates_dict[split])

                file_name = f'hidden_states_{split}_{dim}.csv'
                save_path = os.path.join(round_dir, file_name)

                temp_df.to_csv(save_path, index=False)

    print(f" -> {test_year}년 데이터셋 처리 및 저장 완료")

print("\n 모든 작업 완료")


======== Processing Data for Test Year 2018 ========
Train: 1997 ~ 2014
Valid: 2015 ~ 2017
Test : 2018
 -> 2018년 데이터셋 처리 및 저장 완료

======== Processing Data for Test Year 2019 ========
Train: 1997 ~ 2015
Valid: 2016 ~ 2018
Test : 2019
 -> 2019년 데이터셋 처리 및 저장 완료

======== Processing Data for Test Year 2020 ========
Train: 1997 ~ 2016
Valid: 2017 ~ 2019
Test : 2020
 -> 2020년 데이터셋 처리 및 저장 완료

======== Processing Data for Test Year 2021 ========
Train: 1997 ~ 2017
Valid: 2018 ~ 2020
Test : 2021
 -> 2021년 데이터셋 처리 및 저장 완료

 모든 작업 완료
